# Cross-Lingual Multimodal Transfer Pipeline

Research question: does adding audio help cross-lingual transfer for laughter-associated humor prediction?

Experiment design in this notebook:

- Train on English text only, evaluate on held-out English plus French / Spanish / Hungarian text.
- Train on English text + audio, evaluate on held-out English plus French / Spanish / Hungarian text + audio.

In [1]:
%pip install -U torch transformers librosa soundfile scikit-learn pandas numpy tqdm accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/532.3 MB ? eta -:--:--

   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/532.3 MB 82.4 MB/s eta 0:00:07

   ━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/532.3 MB 132.6 MB/s eta 0:00:04

   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/532.3 MB 134.5 MB/s eta 0:00:04

   ━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/532.3 MB 128.1 MB/s eta 0:00:04

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━ 52/54 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━ 52/54 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━ 52/54 [transformers]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54/54 [accelerate]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyspark-client 4.1.1 requires googleapis-common-protos>=1.71.0, which is not installed.
pyspark-client 4.1.1 requires grpcio>=1.76.0, which is not installed.
pyspark-client 4.1.1 requires grpcio-status>=1.76.0, which is not installed.
pyspark-client 4.1.1 requires pyarrow>=15.0.0, which is not installed.
pyspark-connect 4.1.1 requires googleapis-common-protos>=1.71.0, which is not installed.
pyspark-connect 4.1.1 requires grpcio>=1.76.0, which is not installed.
pyspark-connect 4.1.1 requires grpcio-status>=1.76.0, which is not installed.
pyspark-connect 4.1.1 requires pyarrow>=11.0.0, which is not installed.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

from pipeline.config import PipelineConfig
from pipeline.embeddings import load_or_extract_features
from pipeline.experiment import build_crosslingual_dataset
from pipeline.laughter import AnnotationLaughterDetector
from pipeline.train import (
    evaluate_on_indices,
    metric_row,
    train_transfer_model,
    video_train_val_test_indices,
)

## Config

`RMSLaughterDetector` is only the temporary label module.

In [3]:
cfg = PipelineConfig(
    lang="en",
    # audio_root_name="dataset/audios/en",
    sample_limit=None,  # set to None for the full experiment
    random_seed=42,
    qwen_model_id="Qwen/Qwen3-0.6B",
    whisper_model_id="openai/whisper-large-v3-turbo",
    batch_size_embed_text=4,
    batch_size_train=16,
    epochs=8,
    fusion_hidden_dim=512,
    fusion_num_heads=8,
)

TRAIN_LANG = "en"
TEST_LANGS = ["fr","hu"] # "es", 
ALL_LANGS = [TRAIN_LANG] + TEST_LANGS

AUDIO_ROOTS = {
    "en": cfg.project_root / "cleaned-data" / "en",
    "fr": cfg.project_root / "cleaned-data" / "fr",
    "es": cfg.project_root / "cleaned-data" / "es",
    "hu": cfg.project_root / "cleaned-data" / "hu",
}

SAMPLE_LIMITS = {lang: cfg.sample_limit for lang in ALL_LANGS}

MODEL_MODES = {
    "text_only": "text",
    "text_audio_concat": "concat",
    "text_audio_cross_attention": "cross_attention",
}

random.seed(cfg.random_seed)
np.random.seed(cfg.random_seed)
torch.manual_seed(cfg.random_seed)

print("device:", cfg.device)
print("artifacts:", cfg.artifact_dir)
for lang, root in AUDIO_ROOTS.items():
    print(lang, "transcripts=", cfg.project_root / "asr-output" / lang, "audio=", root)

device: cuda
artifacts: /work/ccs2-train/pipeline_artifacts
en transcripts= /work/ccs2-train/asr-output/en audio= /work/ccs2-train/cleaned-data/en
fr transcripts= /work/ccs2-train/asr-output/fr audio= /work/ccs2-train/cleaned-data/fr
es transcripts= /work/ccs2-train/asr-output/es audio= /work/ccs2-train/cleaned-data/es
hu transcripts= /work/ccs2-train/asr-output/hu audio= /work/ccs2-train/cleaned-data/hu


## Build Cross-Lingual Dataset

In [4]:
laughter_detector = AnnotationLaughterDetector(cfg)
matched_by_lang, segments_df = build_crosslingual_dataset(
    ALL_LANGS,
    AUDIO_ROOTS,
    laughter_detector,
    cfg,
    sample_limits=SAMPLE_LIMITS,
)

segments_csv = cfg.artifact_dir / "crosslingual_segment_dataset.csv"
segments_df.to_csv(segments_csv, index=False)

print("saved:", segments_csv)
print("segments:", segments_df.shape)
print("matched videos by language:")
for lang, matched in matched_by_lang.items():
    print(lang, len(matched))
print("segments by language and label:")
print(pd.crosstab(segments_df["lang"], segments_df["label"]))
segments_df.head()

building segments:   0%|          | 0/158 [00:00<?, ?it/s]

building segments:   0%|          | 0/99 [00:00<?, ?it/s]

building segments:   0%|          | 0/349 [00:00<?, ?it/s]

saved: /work/ccs2-train/pipeline_artifacts/crosslingual_segment_dataset.csv
segments: (74277, 14)
matched videos by language:
en 158
fr 99
hu 349
segments by language and label:
label      0     1
lang              
en     22944  4300
fr     16665  7856
hu     14919  7593


,lang,sample_idx,segment_id,title,json_path,foreground_path,crowd_path,start,end,duration,text,label,crowd_score,num_laughter_intervals_in_video
0,en,0,0,-dr7k9ybQdQ,/work/ccs2-train/asr-output/en/-dr7k9ybQdQ.json,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,0.00,2.18,2.18,And that's how you give a pope a tattoo.,0,0.0,19
1,en,0,1,-dr7k9ybQdQ,/work/ccs2-train/asr-output/en/-dr7k9ybQdQ.json,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,6.92,9.18,2.26,Now we're going to play a game entitled Living...,0,0.0,19
2,en,0,2,-dr7k9ybQdQ,/work/ccs2-train/asr-output/en/-dr7k9ybQdQ.json,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,9.32,11.00,1.68,"This game is for Ryan, Colin, and Wayne.",0,0.0,19
3,en,0,3,-dr7k9ybQdQ,/work/ccs2-train/asr-output/en/-dr7k9ybQdQ.json,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,11.24,12.56,1.32,So come on down.,0,0.0,19
4,en,0,4,-dr7k9ybQdQ,/work/ccs2-train/asr-output/en/-dr7k9ybQdQ.json,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,/work/ccs2-train/cleaned-data/en/-dr7k9ybQdQ F...,13.22,14.72,1.50,You are going to need some help in this game.,0,0.0,19


## English Train/Validation Split

Only English is used for training. English has a held-out in-language test split; French, Spanish, and Hungarian are held out as cross-lingual test languages.

In [5]:
train_idx, val_idx, en_test_idx = video_train_val_test_indices(
    segments_df,
    TRAIN_LANG,
    cfg.random_seed,
    val_size=0.15,
    test_size=0.15,
)

test_indices = {TRAIN_LANG: en_test_idx}
test_indices.update({
    lang: segments_df.index[segments_df["lang"] == lang].to_numpy()
    for lang in TEST_LANGS
})

print("train segments:", len(train_idx), "val segments:", len(val_idx), "en test segments:", len(en_test_idx))
for lang, idx in test_indices.items():
    print(f"test {lang}:", len(idx))

train segments: 18799 val segments: 4811 en test segments: 3634
test en: 3634
test fr: 24521
test hu: 22512


## Extract Or Load Features

The cache checks row identity, so changing language/sample selection will trigger recomputation automatically.

In [6]:
text_hidden, audio_hidden, y = load_or_extract_features(segments_df, cfg, force_recompute=False)

print("num segments:", len(y))
print("text dim:", text_hidden[0].shape[-1], "example text shape:", text_hidden[0].shape)
print("audio dim:", audio_hidden[0].shape[-1], "example audio shape:", audio_hidden[0].shape)
print("labels:", y.shape, np.bincount(y))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

Whisper audio sequence embeddings:   0%|          | 0/74277 [00:00<?, ?it/s]

saved: /work/ccs2-train/pipeline_artifacts/crosslingual_sequence_features_qwen3_whisper.npz
num segments: 74277
text dim: 1024 example text shape: (11, 1024)
audio dim: 1280 example audio shape: (188, 1280)
labels: (74277,) [54528 19749]


## Train On English, Test Cross-Lingually

In [7]:
results = []
histories = {}
trained_models = {}

for model_name, mode in MODEL_MODES.items():
    print("\n===", model_name, "===")
    model, history = train_transfer_model(mode, text_hidden, audio_hidden, y, train_idx, val_idx, cfg)
    trained_models[model_name] = model
    histories[model_name] = history

    for test_lang, indices in test_indices.items():
        metrics = evaluate_on_indices(model, text_hidden, audio_hidden, y, indices, cfg)
        results.append(metric_row(model_name, TRAIN_LANG, test_lang, metrics))

results_df = pd.DataFrame(results).sort_values(["test_lang", "model"])
results_path = cfg.artifact_dir / "crosslingual_transfer_results.csv"
results_df.to_csv(results_path, index=False)
print("saved:", results_path)
results_df



=== text_only ===


           text epoch 01 loss=0.6943 val_f1=0.2283


           text epoch 02 loss=0.6797 val_f1=0.2412


           text epoch 03 loss=0.6715 val_f1=0.2502


           text epoch 04 loss=0.6656 val_f1=0.2461


...

cross_attention epoch 06 loss=0.5609 val_f1=0.1985


cross_attention epoch 07 loss=0.5167 val_f1=0.2219


cross_attention epoch 08 loss=0.4751 val_f1=0.2263


saved: /work/ccs2-train/pipeline_artifacts/crosslingual_transfer_results.csv


,model,train_lang,test_lang,accuracy,precision_1,recall_1,f1_1
3,text_audio_concat,en,en,0.741882,0.322102,0.354599,0.337571
6,text_audio_cross_attention,en,en,0.733076,0.332200,0.434718,0.376607
0,text_only,en,en,0.747386,0.309375,0.293769,0.301370
4,text_audio_concat,en,fr,0.674483,0.472074,0.135565,0.210641
7,text_audio_cross_attention,en,fr,0.646140,0.439765,0.381492,0.408561
1,text_only,en,fr,0.668529,0.435728,0.117363,0.184918
5,text_audio_concat,en,hu,0.645167,0.441377,0.195838,0.271301
8,text_audio_cross_attention,en,hu,0.649476,0.463319,0.247860,0.322952
2,text_only,en,hu,0.640059,0.344891,0.074674,0.122767


## Reading The Result

The key comparison is within each test language:

```text
text_only vs text_audio_concat / text_audio_cross_attention
```

If text+audio improves F1 on held-out English and/or French, Spanish, or Hungarian relative to text-only, that supports the hypothesis that audio helps cross-lingual transfer.

## Replacing The Label Module

```python
class Standup4AILaughterDetector:
    def detect(self, row):
        return [(start_sec, end_sec), ...]
```

Replace:

```python
laughter_detector = AnnotationLaughterDetector(cfg)
```

